# Практическое задание «Проблемы качества данных»

## Шаблон для выполнения

### Шаг 1. Первичный анализ (EDA)

1. Загрузите файл `orders_dirty.csv`. Проведите беглый осмотр данных:

 - Оцените размер таблицы и типы данных.
 - Найдите топ-3 столбца с наибольшим количеством пропусков.
 - Посмотрите уникальные значения в столбцах `payment_method` и `currency`. Есть ли там странные значения?
 - Проверьте, есть ли дубликаты строк по идентификатору заказа.

*В ноутбуке: выведите 2–3 таблицы/графика и напишите краткий вывод (что бросилось в глаза).*

In [12]:
# Загрузите данные и проведите первичный анализ
# импорт базовых библиотек
import numpy as np  # численные операции
import pandas as pd  # таблицы и анализ

pd.set_option('display.max_rows', None) #отменяем ограничение на вывод ячейки
pd.set_option('display.max_columns', None) #отменяем ограничение на вывод ячейки
pd.set_option('display.max_seq_items', None)

# импорт датасета из библиотеки
from sklearn.model_selection import train_test_split  # разбиение train/test
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# импорт графиков
import matplotlib.pyplot as plt  # графики
import seaborn as sns  # статистическая визуализация

# фиксируем генератор случайности
rng = np.random.default_rng(42)

dirty_train = pd.read_csv("orders_dirty.csv")

# размер и типы
print("Размер датасета:", dirty_train.shape)
print("\nТипы столбцов:")
print(dirty_train.dtypes)

dirty_train.head()

# Вывод:получили размер грязного датамета и типы данных. Из всех столбцов 4 являются числовыми (int, float), остальные 8 - строки (str)

Размер датасета: (2570, 12)

Типы столбцов:
row_id              int64
order_id              str
customer_id           str
order_date            str
delivery_date         str
status                str
payment_method        str
currency              str
price             float64
quantity          float64
total_amount      float64
channel               str
dtype: object


,row_id,order_id,customer_id,order_date,delivery_date,status,payment_method,currency,price,quantity,total_amount,channel
0,2211,ORD102211,C79689,2025-08-04,2025-08-05,delivered,paypal,EUR,47.42,4.0,189.68,web
1,2136,ORD102136,C59002,2025-08-25,2025-08-31,delivered,credit_card,EUR,39.94,3.0,119.82,app
2,1178,ORD101178,C33324,2025-01-28,2025-02-04,delivered,credit_card,EUR,29.06,NaN,116.24,web
3,1910,ORD101910,C16688,2024-03-29,2024-04-05,delivered,paypal,EUR,47.86,4.0,191.44,web
4,649,ORD100649,C86369,2024-03-20,2024-03-26,delivered,credit_card,EUR,9.31,5.0,46.55,web


In [13]:
# доля пропусков
print("\nТоп столбцов по пропускам (%):")
print((dirty_train.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))

# уникальные значения категориального столбца
print("\nУникальные payment_method:")
for x in dirty_train["payment_method"].unique():
    print(x)

print("\nУникальные currency:")
for x in dirty_train["currency"].unique():
    print(x)

print("\nУникальные channel:")
for x in dirty_train["channel"].unique():
    print(x)

print("\nКоличество дубликатов order_id:")
print(dirty_train["order_id"].duplicated().sum())

# Вывод: выявили топ 3 столбца с пропусками (точнее 4), а также согласно заданию выявили, что значения признака currency - не создают ложных категорий (значения только EUR и USD), 
# а значения признака payment_method - создают (однако я не в полной мере с этим согласен, так как по сути это все разные значения, которые теоретически нам могут быть полезны)
# Также было выявлено 89 дубликатов значений признака order_id


Топ столбцов по пропускам (%):
order_id          0.78
customer_id       0.78
payment_method    0.58
status            0.58
quantity          0.39
order_date        0.39
price             0.39
row_id            0.00
currency          0.00
delivery_date     0.00
dtype: float64

Уникальные payment_method:
paypal
credit_card
cash
card
crypto
nan
apple_pay

Уникальные currency:
EUR
USD

Уникальные channel:
web
app
partner

Количество дубликатов order_id:
89


In [14]:
print("\nУникальные status:")
for x in dirty_train["status"].unique():
    print(x)

#Вывод: выявили, что значения признака status - создают ложные категории


Уникальные status:
delivered
processing
nan
shipped
done
cancelled
x_cancel
in_progress


### Шаг 2. Реализация детекторов

Вам нужно реализовать 4 функции-детектора. Каждая функция должна возвращать список найденных проблем в формате кортежа: `(индекс_строки, название_колонки, тип_ошибки)`.

**Бизнес-правила для проверки:**

1. **Пропуски (`missing`):** любое значение `NaN` в данных недопустимо.
2. **Дубликаты (`duplicate_row`):** один `order_id` должен встречаться только один раз. Повторы считаются ошибкой.
3. **Нарушения диапазонов (`range_violation`):**
    - Цена (`price`) и количество (`quantity`) не могут быть отрицательными или равными нулю.
4. **Ошибки категорий (`category_inconsistency`):**
    - Статус заказа может быть только: `delivered`, `shipped`, `cancelled`, `processing`.
    - Метод оплаты может быть только: `credit_card`, `paypal`, `cash`.
    - Любые другие значения (включая опечатки, разный регистр) считаются ошибкой.

*Подсказка: используйте множества (set) или списки допустимых значений (whitelist) для проверки категорий.*

In [15]:
def detect_missing(df): #ищем пропуски
    rows, cols = np.where(df.isna())
    return {(int(df.iloc[r]["row_id"]), df.columns[c], "missing") for r, c in zip(rows, cols)}

def detect_duplicates(df): #ищем дубликаты среди order_id (этого будет минимально достаточно для удаления строки)
    dup_idx = df.index[df["order_id"].duplicated(keep="first")]
    return {(int(df.loc[i, "row_id"]), "order_id", "duplicate_row") for i in dup_idx}

def detect_range_violations_price(df):  #ищем нарушения диапазона цены
    mask = df["price"] <= 0
    return {(int(df.loc[i, "row_id"]), "price", "range_violation") for i in df.index[mask]}

def detect_range_violations_qantity(df): #ищем нарушения диапазона количества
    mask = df["quantity"] <= 0
    return {(int(df.loc[i, "row_id"]), "quantity", "range_violation") for i in df.index[mask]}

ALLOWED_SCANNERS_STATUS = {"delivered", "shipped", "cancelled", "processing"} #допустимые «чистые» категории статуса
def detect_category_inconsistency_status(df): #ищем нарушения категорийности в статусе
    mask = ~df["status"].isin(ALLOWED_SCANNERS_STATUS)
    return {(int(df.loc[i, "row_id"]), "status", "category_inconsistency") for i in df.index[mask]}

ALLOWED_SCANNERS_METHOD = {"credit_card", "paypal", "cash"} #допустимые «чистые» категории метода оплаты
def detect_category_inconsistency_method(df): #ищем нарушения категорийности в методе оплаты
    mask = ~df["payment_method"].isin(ALLOWED_SCANNERS_METHOD)
    return {(int(df.loc[i, "row_id"]), "payment_method", "category_inconsistency") for i in df.index[mask]}

#запускаем все детекторы
detected_set = set().union(
    detect_missing(dirty_train),
    detect_duplicates(dirty_train),
    detect_range_violations_price(dirty_train),
    detect_range_violations_qantity(dirty_train),
    detect_category_inconsistency_status(dirty_train),
    detect_category_inconsistency_method(dirty_train))

# переводим в DataFrame
detected_df = pd.DataFrame(list(detected_set), columns=["row_id", "column", "issue_type"])

# сколько нашли по типам
detected_df["issue_type"].value_counts()

# Вывод: наша проверка обнаружила 364 ошибки

issue_type
missing                   100
category_inconsistency    100
duplicate_row              89
range_violation            75
Name: count, dtype: int64

### Шаг 3. Оценка качества поиска

1. Загрузите файл `orders_truth_hidden.csv`. Сравните список ошибок, который нашли ваши детекторы, с эталонным списком.
2. Рассчитайте метрику **Recall** по формуле:

$$ Recall = \frac{TP}{TP + FN} $$

где:
- **TP (True Positive)**: количество совпавших ошибок (найденных вами и существующих в эталоне).
- **TP + FN**: общее количество ошибок в файле truth.

**Цель:** получить общий Recall не ниже 80%.

In [16]:
# формируем множества «истина» и «нашли»
df_true = pd.read_csv("orders_truth_hidden.csv")

true_set = set(df_true.itertuples(index=False, name=None))
found_set = set(detected_df.itertuples(index=False, name=None))

# считаем TP и recall
tp_set = true_set & found_set
recall = len(tp_set) / len(true_set)

# печатаем итог
print(f"Всего истинных проблем: {len(true_set)}")
print(f"Найдено корректно (TP): {len(tp_set)}")
print(f"Recall: {recall:.2%}")

# Рассчитайте Recall
# Вывод: видим, что Recall = 81,82%, чего достаточно согласно заданию, однако выясним, где недостающие 18%


Всего истинных проблем: 385
Найдено корректно (TP): 315
Recall: 81.82%


In [17]:
missed = true_set - found_set

pd.DataFrame(list(missed),columns=["row_id", "column", "issue_type"]).sort_values("issue_type")

#Видим, что все недостающие данные - дубликаты строк. 
#Обратим внимание, что истинный файл считает ошибкой - первое вхождение дубликата. 

,row_id,column,issue_type
0,2515,order_id,duplicate_row
37,1259,order_id,duplicate_row
38,1778,order_id,duplicate_row
39,1099,order_id,duplicate_row
40,1976,order_id,duplicate_row
41,2519,order_id,duplicate_row
42,2527,order_id,duplicate_row
43,460,order_id,duplicate_row
44,1921,order_id,duplicate_row
45,438,order_id,duplicate_row


In [18]:
#Тогда исправленный код (ниже) - изменение пометил хэштегом, остальное - без изменений
def detect_missing(df):
    rows, cols = np.where(df.isna())
    return {(int(df.iloc[r]["row_id"]), df.columns[c], "missing") for r, c in zip(rows, cols)}
def detect_duplicates(df):
    dup_idx = df.index[df["order_id"].duplicated(keep=False)] #Меняем keep="first" на keep=False
    return {(int(df.loc[i, "row_id"]), "order_id", "duplicate_row") for i in dup_idx}
def detect_range_violations_price(df):
    mask = df["price"] <= 0
    return {(int(df.loc[i, "row_id"]), "price", "range_violation") for i in df.index[mask]}
def detect_range_violations_qantity(df):
    mask = df["quantity"] <= 0
    return {(int(df.loc[i, "row_id"]), "quantity", "range_violation") for i in df.index[mask]}
ALLOWED_SCANNERS_STATUS = {"delivered", "shipped", "cancelled", "processing"}
def detect_category_inconsistency_status(df):
    mask = ~df["status"].isin(ALLOWED_SCANNERS_STATUS)
    return {(int(df.loc[i, "row_id"]), "status", "category_inconsistency") for i in df.index[mask]}
ALLOWED_SCANNERS_METHOD = {"credit_card", "paypal", "cash"}
def detect_category_inconsistency_method(df):
    mask = ~df["payment_method"].isin(ALLOWED_SCANNERS_METHOD)
    return {(int(df.loc[i, "row_id"]), "payment_method", "category_inconsistency") for i in df.index[mask]}
detected_set = set().union(
    detect_missing(dirty_train),
    detect_duplicates(dirty_train),
    detect_range_violations_price(dirty_train),
    detect_range_violations_qantity(dirty_train),
    detect_category_inconsistency_status(dirty_train),
    detect_category_inconsistency_method(dirty_train))
detected_df = pd.DataFrame(list(detected_set), columns=["row_id", "column", "issue_type"])
detected_df["issue_type"].value_counts()
df_true = pd.read_csv("orders_truth_hidden.csv")
true_set = set(df_true.itertuples(index=False, name=None))
found_set = set(detected_df.itertuples(index=False, name=None))
tp_set = true_set & found_set
recall = len(tp_set) / len(true_set)
print(f"Всего истинных проблем: {len(true_set)}")
print(f"Найдено корректно (TP): {len(tp_set)}")
print(f"Recall: {recall:.2%}")

#Вывод - теперь мы обнаружили 100% истинных проблем.

Всего истинных проблем: 385
Найдено корректно (TP): 385
Recall: 100.00%


In [19]:
#Проверим, есть ли у нас неистинные проблемы
missed = found_set - true_set

pd.DataFrame(list(missed),columns=["row_id", "column", "issue_type"]).sort_values("issue_type")

,row_id,column,issue_type
0,369,status,category_inconsistency
48,1291,status,category_inconsistency
25,145,payment_method,category_inconsistency
26,517,payment_method,category_inconsistency
28,1704,status,category_inconsistency
29,285,status,category_inconsistency
30,1357,status,category_inconsistency
33,2348,payment_method,category_inconsistency
34,1506,payment_method,category_inconsistency
37,445,status,category_inconsistency


In [20]:
# Мы имеем 2 вида проблем: 
# 1)если NaN попадается в статус заказа или метод оплаты, то ошибка "дублируется"
# 2)если NaN попадается в order_id, то детектор показывает его "дубликатом"
# Исправим код в связи с обнаруженным дублированием - изменение пометил хэштегом, остальное - без изменений
def detect_missing(df):
    rows, cols = np.where(df.isna())
    return {(int(df.iloc[r]["row_id"]), df.columns[c], "missing") for r, c in zip(rows, cols)}
def detect_duplicates(df):
    mask = ((df["order_id"].notna()) & (df["order_id"].duplicated(keep=False))) #исключаем пустые значения из категории "дубликаты"
    return {(int(df.loc[i, "row_id"]), "order_id", "duplicate_row") for i in df.index[mask]}
def detect_range_violations_price(df):
    mask = df["price"] <= 0
    return {(int(df.loc[i, "row_id"]), "price", "range_violation") for i in df.index[mask]}
def detect_range_violations_qantity(df):
    mask = df["quantity"] <= 0
    return {(int(df.loc[i, "row_id"]), "quantity", "range_violation") for i in df.index[mask]}
ALLOWED_SCANNERS_STATUS = {"delivered", "shipped", "cancelled", "processing"}
def detect_category_inconsistency_status(df): #добавили проверку на пустые значения
    mask = ((~df["status"].isin(ALLOWED_SCANNERS_STATUS)) & (df["status"].notna()))
    return {(int(df.loc[i, "row_id"]), "status", "category_inconsistency") for i in df.index[mask]}
ALLOWED_SCANNERS_METHOD = {"credit_card", "paypal", "cash"}
def detect_category_inconsistency_method(df): #добавили проверку на пустые значения
    mask = ((~df["payment_method"].isin(ALLOWED_SCANNERS_METHOD)) & (df["payment_method"].notna()))
    return {(int(df.loc[i, "row_id"]), "payment_method", "category_inconsistency") for i in df.index[mask]}
detected_set = set().union(
    detect_missing(dirty_train),
    detect_duplicates(dirty_train),
    detect_range_violations_price(dirty_train),
    detect_range_violations_qantity(dirty_train),
    detect_category_inconsistency_status(dirty_train),
    detect_category_inconsistency_method(dirty_train))
detected_df = pd.DataFrame(list(detected_set), columns=["row_id", "column", "issue_type"])
detected_df["issue_type"].value_counts()
df_true = pd.read_csv("orders_truth_hidden.csv")
true_set = set(df_true.itertuples(index=False, name=None))
found_set = set(detected_df.itertuples(index=False, name=None))
tp_set = true_set & found_set
recall = len(tp_set) / len(true_set)
print(f"Всего истинных проблем: {len(true_set)}")
print(f"Найдено корректно (TP): {len(tp_set)}")
print(f"Recall: {recall:.2%}")
missed = found_set - true_set
pd.DataFrame(list(missed), columns=["row_id", "column", "issue_type"]).sort_values("issue_type")

#Вот теперь мы определили 100% истинных ошибок и 0% ложных.

Всего истинных проблем: 385
Найдено корректно (TP): 385
Recall: 100.00%


,row_id,column,issue_type


### Шаг 4. Очистка и Моделирование

Докажите бизнесу, что чистка данных имеет смысл.

1. **Напишите функцию очистки:**
    - Удалите дубликаты.
    - Заполните пропуски (например, медианой для чисел и модой для категорий).
    - Исправьте опечатки в категориях (приведите к нижнему регистру, устраните неявные дубликаты).
    - Обработайте некорректные числовые значения (например, возьмите модуль от отрицательной цены).

2. **Обучите модель:**
    - Подготовьте данные: целевая метка формируется из `status`: `target=1`, если `status='cancelled'`, иначе `0`.
    - Обучите простую `LogisticRegression` дважды: на «грязных» данных и на очищенных.
    - Сравните метрику `Accuracy`.

In [21]:
# Функция очистки данных
from datetime import datetime
def clean_data(df):
    clean = df.copy()

    clean = clean.drop_duplicates(subset=["order_id"], keep="first").reset_index(drop=True) #удаляем дубликаты

    clean["status"] = clean["status"].str.strip() #чистим и нормализуем категории
    cat_map = {
        "delivered": "delivered", "done": "delivered",
        "processing": "processing", "in_progress": "processing",
        "cancelled": "cancelled", "x_cancel": "cancelled",
        "shipped": "shipped"
        }
    clean["status"] = clean["status"].replace(cat_map)
    clean.loc[~clean["status"].isin(ALLOWED_SCANNERS_STATUS), "status"] = np.nan


    clean["payment_method"] = clean["payment_method"].str.strip() #чистим и нормализуем категории
    cat_map = {
        "credit_card": "credit_card", "card": "credit_card",
        "paypal": "paypal", "apple_pay": "paypal", "crypto": "paypal",
        "cash": "cash"
        }
    clean["payment_method"] = clean["payment_method"].replace(cat_map)
    clean.loc[~clean["payment_method"].isin(ALLOWED_SCANNERS_METHOD), "payment_method"] = np.nan
    clean["payment_method"] = clean["payment_method"].fillna(clean["payment_method"].mode()[0])

    clean.loc[clean["price"] <= 0, "price"] = np.nan #некорректный диапазон -> NaN
    clean.loc[clean["quantity"] <= 0, "quantity"] = np.nan #некорректный диапазон -> NaN

    #заполняем числовые пропуски медианой
    num_cols = clean.select_dtypes(include="number").columns.tolist() 
    clean[num_cols] = clean[num_cols].fillna(clean[num_cols].median())

    #заполняем строковые пропуски модой
    cat_cols = clean.select_dtypes(include=["object", "string"]).columns.tolist()   
    if "status" in cat_cols:
        cat_cols.remove("status") #статус удаляем, чтобы функция не заменила пропуски целевой переменной модой
    clean[cat_cols] = clean[cat_cols].fillna(clean[cat_cols].mode().iloc[0])

    clean = clean.dropna(subset=["status"]) # удаляем пропуски в статусе заказа, так как это целевая переменная

    #работаем с датами заказа и доставки
    clean["delivery_date"] = pd.to_datetime(clean["delivery_date"], format="%Y-%m-%d")
    clean["order_date"] = pd.to_datetime(clean["order_date"], format="%Y-%m-%d")
    clean["delivery_days"]=(clean["delivery_date"]-clean["order_date"]).dt.days
    mask = clean["delivery_date"] < clean["order_date"]
    median_days = clean.loc[~mask, "delivery_days"].median()
    clean.loc[mask, "delivery_days"] = median_days

    return clean

# применяем очистку
clean_train = clean_data(dirty_train)

# быстрые проверки после очистки
post_check = {
    "rows": len(clean_train),
    "missing_total": int(clean_train.isna().sum().sum()),
    "duplicate_rows": int(clean_train.duplicated().sum()),
    "range_violations_price": int((clean_train["price"] <= 0).sum()),
    "range_violations_mean_quantity": int((clean_train["quantity"] <= 0).sum()),
    "bad_categories_status": int((~clean_train["status"].isin(ALLOWED_SCANNERS_STATUS)).sum()),
    "bad_categories_payment_method": int((~clean_train["payment_method"].isin(ALLOWED_SCANNERS_METHOD)).sum())
}
pd.Series(post_check)

rows                              2466
missing_total                        0
duplicate_rows                       0
range_violations_price               0
range_violations_mean_quantity       0
bad_categories_status                0
bad_categories_payment_method        0
dtype: int64

In [22]:
# Обучение моделей и сравнение Accuracy
dirty = dirty_train.copy() #копируем датасет "грязный" перед его изменением (минимальной подготовкой для обучения модели (не равно "очистка"))

dirty["status"] = np.where(dirty["status"] == "cancelled", 1, 0) #в грязном датасете там где у нас отмена (даже с ошибкой) - ставим 1, где нет - 0. Для обучения модели

clean_train["status"] = np.where(clean_train["status"] == "cancelled", 1, 0) #то же самое в чистом датасете

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler #импортирую метод, который позволит учесть для обучения столбцы с нечисловыми значениями
from sklearn.pipeline import Pipeline

numeric_columns_dirty = ["price","quantity","total_amount"] #создаем список числовых значений для грязных данных (без длительности доставки)
numeric_columns_clean = ["price","quantity","total_amount","delivery_days"] #создаем список числовых значений для чистых данных (с длительностью доставки)
categorical_columns = ["payment_method","currency","channel"] #создаем список нечисловых значений
numeric_columns=[]

def get_acc(df, use_delivery_days=False):
    if use_delivery_days:
        numeric_columns = numeric_columns_clean
    else:
        numeric_columns = numeric_columns_dirty
    X = df.drop(["status", "row_id", "order_id", "customer_id"], axis=1) #исключаем столбцы, по смыслу бизнес-процесса не являющиеся признаками (+целевую переменную)
    X[numeric_columns] = X[numeric_columns].fillna(-1) #в числовых столбцах заменяем пропуски на -1. БЕЗ ЭТОГО ОБУЧЕНИЕ МОДЕЛИ НА ГРЯЗНЫХ ДАННЫХ ПАДАЕТ В ОШИБКУ, ЧТО ЯВЛЯЕТСЯ ДОКАЗАТЕЛЬСТВОМ НЕОБХОДИМОСТИ ОЧИСТКИ ДАТАСЕТА
    y = df["status"] #выбираем целевую переменную

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42) #отделяем 20% данных на тест

    #создаем объект, который будет по разному обрабатывать разные группы столбцов перед обучением модели
    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), numeric_columns),   #масштабирование числовых признаков 
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)]) #превращение категориальных значений в 1 и 0. handle_unknown="ignore" - если модель увидит категорию, не попавшую в обучение - ставить 0

    #создаем модель, которая выполняет последовательность действий. 
    #cперва препроцессор - перевод нечисловых столбцов в числовые (OneHotEncoder), а числовых в (StandardScaler) - масштабирование. Масштабирование категорийных (в прошлом) столбцов не требуется.
    #После этого общая таблица идет в модель LogisticRegression (до 5000 итераций)
    model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=5000, random_state=42))])  

    model.fit(X_train, y_train) #обучаем модель
    return accuracy_score(y_test, model.predict(X_test)) #возвращаем точность предсказанных данных

acc_dirty = get_acc(dirty)
acc_clean = get_acc(clean_train, use_delivery_days=True)

print(f"Точность на грязных данных: {acc_dirty:.1%}")
print(f"Точность на чистых данных:  {acc_clean:.1%}")
print(f"Прирост качества:           {acc_clean - acc_dirty:+.1%}")

print(f"\nРаспределение целевого параметра: {dirty["status"].value_counts(normalize=True)}")

Точность на грязных данных: 90.1%
Точность на чистых данных:  90.9%
Прирост качества:           +0.8%

Распределение целевого параметра: status
0    0.906615
1    0.093385
Name: proportion, dtype: float64


### Шаг 5. Итоги (рефлексия)

Напишите выводы (5–7 предложений):

1. Какие ошибки было найти сложнее всего?
2. Как сильно выросла точность модели после очистки?
3. Какую **одну** проверку (из реализованных вами) вы бы поставили на ежедневный мониторинг в первую очередь и почему?

1. В отличие от пропусков, дубликатов и нарушений диапазонов значений гораздо сложнее работать со строковыми данными, а также у меня заняло продолжительное время обнаружение того, что мой "счетчик" ошибок дублирует ошибки, из-за чего я не достиг 100% совпадения с образцом изначально.
2. Точность модели после очистки данных выросла на 0,8%, однако это значение не дает нам право говорить о повышении точности, в связи с тем, что меняя значения random_state и test_size - можно получить совершенно иные данные, в том числе и отрицательные. А если при отборе данных на test и train еще и сохранить распределение (stratify=y), то прирост вообще будет как правило отрицательным. Думаю причина кроется в самом датасете, а именно в распределении долей мажорного и минорного таргета (10% против 90%). То есть мы учим модель предсказывать то, что заказ не отменят. По моему мнению именно это мешает нам увидеть уверенное увеличение точности модели после очистки данных. (Очень хотелось бы услышать обратную связь по поводу данной гипотезы). Хотя о необходимости подготовки данных и их очистке, говорит хотя бы тот факт, что нам пришлось грязный датасет чистить (X[numeric_columns] = X[numeric_columns].fillna(-1)). Без этого код падал в ошибку.
3. Сложно выделить одну, так как все проверки кажутся максимально логичными и нужными для обучения модели, однако, если все-таки выбирать, я бы отдал предпочтение либо нарушению диапазона, либо нарушению классификации. Так как в моем понимании они сильнее могут навредить модели по сравнению с дубликатами, с пропусками.


...